# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step usage of [`mlcroissant`](https://mlcommons.github.io/croissant/python/latest/) to load, review, process, and visualize the FAIR² colorectal cancer survivors dataset defined by its [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully referenced using entity `@id`s for robust access and exploration.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Let's load the dataset metadata and records using `mlcroissant`.

The dataset Croissant JSON-LD schema is given by:
```python
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
```


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset ID: {metadata.id}")
print(f"Published: {metadata.date_published}\n")

## 2. Data Overview

Let's review available record sets and their fields. 

`mlcroissant` represents each data container in the schema as a **record set**, and each set contains **fields** (columns).

We will enumerate all record sets and for each, print the fields' `@id` and names.

In [ ]:
# List all Record Sets with their @id and field information

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the schema.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for r in record_sets:
        print(f"Record Set: {r.name}")
        print(f"  @id: {r.id}")
        if r.fields:
            for f in r.fields:
                print(f"    Field: {f.name} (@id: {f.id}, dataType: {getattr(f, 'data_type', None)})")
        else:
            print("    (No fields found)")
        print("-")

## 3. Data Extraction

We will load each available record set using its `@id`, and create a pandas DataFrame for each set.

**Note**: If there is only one record set in this dataset, it will be loaded as the primary DataFrame.


In [ ]:
# Identify all record set @ids for extraction
record_sets = dataset.record_sets
record_set_ids = [r.id for r in record_sets]

# Demonstrate extraction of all record sets into DataFrames
dataframes = {}
for rec_id in record_set_ids:
    # Load all records into a list of dicts
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records from record set '@id': {rec_id}")

# For convenience, let's use the first record set (if any) for further analysis
if len(record_set_ids) > 0:
    first_id = record_set_ids[0]
    print(f"\nSample columns for record set '@id': {first_id}")
    print(dataframes[first_id].columns.tolist())
    print("\nSample data:")
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform common EDA steps:
- Filtering (e.g. age or diagnosis interval)
- Normalization (z-score of numeric fields)
- Grouping (anatomic distribution, sex, MSI-H status, etc.)

> **Note:** All columns/fields are referenced by their `@id`. If unsure which fields are numeric, check above for the fields' dataType.


In [ ]:
# Select an example record set and numeric field by @id
if len(record_set_ids) > 0:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Choose a likely numeric field: try 'age' or similar
    # You may need to adapt the field @id based on the actual schema.
    probable_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower()]
    if probable_numeric_fields:
        numeric_field = probable_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        # Coerce to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.5)  # median as example filter
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nRecords where {numeric_field} > {threshold:.2f} (median): {len(filtered_df)}")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nFirst 5 normalized {numeric_field} values:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by another column (e.g. sex, anatomical location)
        group_fields = [col for col in df.columns if any(keyword in col.lower() for keyword in ['sex', 'anatomic', 'site', 'location'])]
        group_field = group_fields[0] if group_fields else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable field found to group by.")
    else:
        print("No obvious numeric field found to demo filtering and normalization.")
else:
    print("No record sets available for EDA.")

## 5. Visualization

We now visualize the distribution of a numeric variable and its relationship to a categorical attribute using matplotlib and seaborn.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the previously selected numeric field
if len(record_set_ids) > 0 and 'numeric_field' in locals():
    # Distribution plot
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If we also have a group field, show boxplot
    if 'group_field' in locals() and group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded metadata and tables from the FAIR² colorectal cancer survivors dataset using the Croissant schema and `mlcroissant`.
- Enumerated all record sets and presented their fields by `@id`.
- Loaded record set(s) as pandas DataFrame, performed numeric filtering and normalization by field `@id`.
- Grouped and visualized data based on key clinical features to support reproducible biomedical analyses.

Refer to the [dataset schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/) for further advanced exploration or model-ready data pipelines.